In [0]:
%run ./01.config.py

In [0]:


# COMMAND ----------
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

cols = ["transaction_id","transaction_date","transaction_year","transaction_quarter",
        "transaction_month","product_name","product_category","quantity_unit",
        "supplier_name","supplier_country","supplier_reliability_score","refinery_name",
        "destination_city","transportation_mode","ordered_quantity","demand_quantity",
        "available_inventory","unit_price_usd","product_cost_usd","transportation_cost_usd",
        "total_cost_usd","expected_lead_time_days","actual_lead_time_days","delay_days",
        "is_delayed","is_stockout","quality_status","quality_score","disruption_type",
        "delivery_status","ingestion_timestamp","source_system","operation_type"]
schema = StructType([StructField(c, StringType(), True) for c in cols])

df = (spark.read.option("header", True).option("multiLine", True)
      .option("rescuedDataColumn", "_rescued_data")
      .schema(schema).csv(SRC_FILE))

df = df.withColumn("txn_date", F.coalesce(
        F.expr("try_to_date(transaction_date, 'yyyy-MM-dd')"),
        F.expr("try_to_date(transaction_date, 'dd-MM-yyyy')"),
        F.expr("try_to_date(transaction_date, 'MM/dd/yyyy')"),
        F.expr("try_to_date(transaction_date, 'dd/MM/yyyy')"),
        F.to_date(F.expr("try_to_timestamp(transaction_date)"))))

df = (df.withColumn("_ingest_ts", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path")))

df.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(BRONZE)

# COMMAND ----------
b = spark.table(BRONZE)
print("rows:", b.count(), "| null txn_date:", b.filter("txn_date IS NULL").count())
display(b.limit(100000))